In [1]:
# Install the latest torch-judge from this repo in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q --force-reinstall --no-deps git+https://github.com/CharlesShang/TorchCode.git@master')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.0 MB/s eta 0:00:00


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CharlesShang/TorchCode/blob/master/templates/28_moe.ipynb)

# 🔴 Hard: Mixture of Experts (MoE)

Implement a **Mixture of Experts** layer (Mixtral / Switch Transformer style).

### Signature
```python
class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2): ...
    def forward(self, x: Tensor) -> Tensor:
        # x: (B, S, D) -> (B, S, D)
```

### Architecture
- `self.router`: `nn.Linear(d_model, num_experts)` — gating network
- `self.experts`: `nn.ModuleList` of MLPs `(Linear→ReLU→Linear)`
- For each token: select top-k experts, compute weighted sum of their outputs

In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F

help(torch.topk)

Help on built-in function topk in module torch:

topk(...)
    topk(input, k, dim=None, largest=True, sorted=True, *, out=None) -> (Tensor, LongTensor)

    Returns the :attr:`k` largest elements of the given :attr:`input` tensor along
    a given dimension.

    If :attr:`dim` is not given, the last dimension of the `input` is chosen.

    If :attr:`largest` is ``False`` then the `k` smallest elements are returned.

    A namedtuple of `(values, indices)` is returned with the `values` and
    `indices` of the largest `k` elements of each row of the `input` tensor in the
    given dimension `dim`.

    The boolean option :attr:`sorted` if ``True``, will make sure that the returned
    `k` elements are themselves sorted

    .. note::
        When using `torch.topk`, the indices of tied elements are not guaranteed to be stable
        and may vary across different invocations.

    Args:
        input (Tensor): the input tensor.
        k (int): the k in "top-k"
        dim (int, option

In [29]:
# ✏️ YOUR IMPLEMENTATION HERE

class MLP(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )
    def forward(self, x):
        return self.mlp(x)

class MixtureOfExperts(nn.Module):
    def __init__(self, d_model, d_ff, num_experts, top_k=2):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.router = nn.Linear(d_model, num_experts)
        self.experts = nn.ModuleList(
            [MLP(d_model, d_ff) for _ in range(num_experts)]
        )

    def forward(self, x):

        B, S, D = x.shape
        x_flat = x.reshape(B * S, D)
        T = x_flat.shape[0]
        # router_logits: (T, E)
        router_logits = self.router(x_flat)
        # router_probs: (T, E)
        router_probs = F.softmax(router_logits, dim=-1)

        # --- export selection using topk
        # topk_probs:   (T, K)
        # topk_indices: (T, K)
        topk_probs, topk_indices = torch.topk(
            router_probs,
            k=self.top_k,
            dim=-1,
        )
        # topk_probs: (T, K)
        topk_probs = topk_probs / topk_probs.sum(dim=-1, keepdim=True)

        # build output tensor first (T, D)
        y_flat = torch.zeros_like(x_flat)

        for expert_id, expert in enumerate(self.experts):
            mask = topk_indices == expert_id
            token_mask = mask.any(dim=-1)

            # print(f"{expert_id=} {token_mask.shape=} {mask.shape=} {x_flat.shape=}")

            # -- forward on seleted tokens
            selected_x = x_flat[token_mask]
            expert_out = expert(selected_x)

            print(f"{expert_id=} {selected_x.shape=}")

            # apply weighted sum
            expert_weight = (
                  topk_probs[token_mask] * mask[token_mask].float()
            ).sum(dim=-1)
            # print(f"{expert_id=} {expert_weight.shape=}")
            y_flat[token_mask] += expert_out * expert_weight.unsqueeze(-1)

        return y_flat.reshape(B, S, D)



In [30]:
# 🧪 Debug
moe = MixtureOfExperts(32, 64, num_experts=4, top_k=2)
x = torch.randn(2, 8, 32)
print('Output:', moe(x).shape)
print('Params:', sum(p.numel() for p in moe.parameters()))

expert_id=0 token_mask.shape=torch.Size([16]) mask.shape=torch.Size([16, 2]) x_flat.shape=torch.Size([16, 32])
expert_id=1 token_mask.shape=torch.Size([16]) mask.shape=torch.Size([16, 2]) x_flat.shape=torch.Size([16, 32])
expert_id=2 token_mask.shape=torch.Size([16]) mask.shape=torch.Size([16, 2]) x_flat.shape=torch.Size([16, 32])
expert_id=3 token_mask.shape=torch.Size([16]) mask.shape=torch.Size([16, 2]) x_flat.shape=torch.Size([16, 32])
Output: torch.Size([2, 8, 32])
Params: 16900


In [23]:
# ✅ SUBMIT
from torch_judge import check
check('moe')


🧪 Testing: Mixture of Experts (MoE) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (4.8ms)
  ✅ [2/4] Has router and experts (0.8ms)
  ✅ [3/4] Router logits shape (1.5ms)
  ✅ [4/4] Gradient flow (2.4ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (9.5ms total)
  Progress saved. Run status() to see your dashboard.

